# 01 - Exploração do banco

Primeiro contato com a tabela `ocorrencias` (Sinesp VDE, jan a out de 2026), antes de qualquer análise. O objetivo é entender:

- o que cada linha representa;
- quais eventos existem e em que nível (município ou estado);
- onde há nulos, zeros e buracos de cobertura.

Nada aqui compara regiões: sem população, contagem bruta não diz qual lugar é mais violento.

## 1. Conexão e tamanho

Abrimos o SQLite e carregamos a tabela inteira no pandas. Com cerca de 675 mil linhas, ela cabe na memória.

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

caminho_banco = Path("..") / "data" / "db" / "crimes.db"
conexao = sqlite3.connect(caminho_banco)
print(caminho_banco.resolve(), caminho_banco.exists())

In [ ]:
esquema = pd.read_sql("SELECT sql FROM sqlite_master WHERE type IN ('table', 'index')", conexao)
for comando in esquema["sql"]:
    print(comando, end="\n\n")

In [ ]:
df = pd.read_sql("SELECT * FROM ocorrencias", conexao)
print(df.shape)
df.head()

In [ ]:
print(df.dtypes)
print("Memória (MB):", round(df.memory_usage(deep=True).sum() / 1e6, 1))

Período coberto e número de linhas por mês. Uma diferença de linhas entre meses indica que algum evento ou UF não veio naquele mês.

In [ ]:
print("De", df["data_referencia"].min(), "a", df["data_referencia"].max())
print("Anos:", df["ano"].unique())
print(df["mes"].value_counts().sort_index())

In [ ]:
print("UFs:", df["uf"].nunique())
print("Pares (uf, municipio):", df[["uf", "municipio"]].drop_duplicates().shape[0])
print("Pares sem 'NÃO INFORMADO':", df.loc[df["municipio"] != "NÃO INFORMADO", ["uf", "municipio"]].drop_duplicates().shape[0])
print(df["abrangencia"].value_counts())

## 2. Catálogo de eventos

Para cada combinação de evento e abrangência, a tabela mostra:

- **pares_uf_municipio**: quantos pares distintos (uf, municipio) o evento tem. Contar só o nome do município daria um número menor, porque há nomes repetidos entre UFs (seção 8);
- **nivel**: se o evento é aberto por município ou só vem por estado (`municipio = 'NÃO INFORMADO'`);
- **metrica**: o que o número significa. No Excel cada evento usa uma só das colunas `total_vitima`, `total` ou `total_peso`, e o `carga.py` grava essa origem na coluna `metrica`:
  - **vítimas**: pessoas atingidas (coluna `total_vitima`);
  - **total**: ocorrências ou procedimentos, conforme o evento (mandados cumpridos, alvarás emitidos etc.);
  - **peso**: apreensões de drogas, na coluna `peso`. A unidade não é informada no arquivo.

Este catálogo vai servir também para descrever o banco ao LLM no notebook de GenAI.

In [ ]:
catalogo = (
    df.groupby(["evento", "abrangencia", "metrica"])
    .agg(
        linhas=("uf", "size"),
        ufs=("uf", "nunique"),
        meses=("mes", "nunique"),
        quantidade=("quantidade", lambda s: s.sum(min_count=1)),
        peso=("peso", lambda s: s.sum(min_count=1)),
    )
    .reset_index()
)
pares_por_evento = df[["evento", "abrangencia", "uf", "municipio"]].drop_duplicates().groupby(["evento", "abrangencia"]).size()
catalogo["pares_uf_municipio"] = pares_por_evento.values
catalogo["nivel"] = catalogo["pares_uf_municipio"].gt(catalogo["ufs"]).map({True: "municipal", False: "estadual"})
print(catalogo.shape)
catalogo

In [ ]:
print(catalogo.groupby(["nivel", "metrica"]).size())
print()
print("Eventos distintos:", catalogo["evento"].nunique())
print("Eventos com mais de uma abrangência:")
print(catalogo.groupby("evento")["abrangencia"].nunique().loc[lambda s: s > 1])

## 3. Nulos

Muitos nulos são esperados: cada coluna de detalhe só existe para alguns eventos. Por exemplo, `arma` só faz sentido em "Arma de Fogo Apreendida". A segunda tabela mostra, por evento, quantas linhas têm cada coluna preenchida.

In [ ]:
nulos = pd.DataFrame({"nulos": df.isna().sum(), "pct": (df.isna().mean() * 100).round(1)})
nulos

In [ ]:
colunas_detalhe = ["agente", "arma", "faixa_etaria", "feminino", "masculino", "nao_informado", "quantidade", "peso"]
preenchidas = df[colunas_detalhe].notna().groupby(df["evento"]).sum()
preenchidas

Nas linhas com sexo preenchido, conferimos se `feminino + masculino + nao_informado` bate com `quantidade`.

In [ ]:
sexo = df.dropna(subset=["feminino", "masculino", "nao_informado"]).copy()
sexo["soma_sexo"] = sexo["feminino"] + sexo["masculino"] + sexo["nao_informado"]
diferentes = sexo[sexo["soma_sexo"] != sexo["quantidade"]]
print("Linhas com sexo preenchido:", len(sexo))
print("Linhas onde a soma por sexo difere de quantidade:", len(diferentes))
diferentes.groupby("evento").size()

## 4. Cobertura por UF e mês

Contamos quantas UFs têm linha em cada mês, por evento. O esperado é 27 em todos os meses. Valores menores indicam UF que não informou ou mês que ainda não chegou.

Atenção: ausência de linha é diferente de zero. Uma UF que não informou não pode entrar numa média como se tivesse zero ocorrências.

In [ ]:
cobertura = df.pivot_table(index=["evento", "abrangencia"], columns="mes", values="uf", aggfunc="nunique").fillna(0).astype(int)
incompletos = cobertura[(cobertura < 27).any(axis=1)]
print("Combinações evento/abrangência:", len(cobertura))
print("Com algum mês abaixo de 27 UFs:", len(incompletos))
incompletos

In [ ]:
todas_ufs = set(df["uf"])
ufs_ausentes = df.groupby(["evento", "abrangencia"])["uf"].apply(lambda s: sorted(todas_ufs - set(s)))
ufs_ausentes[ufs_ausentes.str.len() > 0]

In [ ]:
meses_por_evento = df.groupby(["evento", "abrangencia"])["mes"].agg(["min", "max", "nunique"])
meses_por_evento[meses_por_evento["nunique"] < 10]

## 5. Zeros e "NÃO INFORMADO"

Nos eventos municipais, a base parece trazer uma linha para todo município em todo mês, mesmo sem ocorrência. A proporção de zeros mostra isso. Em municípios pequenos, isso deixa a série muito irregular: um único caso muda muito a taxa.

Também vemos quanto de cada evento municipal ficou na linha sem município, que não pode ser atribuído a nenhuma cidade.

In [ ]:
municipais = catalogo.loc[catalogo["nivel"] == "municipal", ["evento", "abrangencia"]]
base_municipal = df.merge(municipais, on=["evento", "abrangencia"])
zeros = (
    base_municipal.assign(zero=base_municipal["quantidade"].eq(0))
    .groupby(["evento", "abrangencia"])["zero"]
    .mean()
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
)
print("Linhas nos eventos municipais:", len(base_municipal))
zeros

In [ ]:
total = base_municipal.groupby(["evento", "abrangencia"])["quantidade"].sum()
sem_municipio = base_municipal[base_municipal["municipio"] == "NÃO INFORMADO"].groupby(["evento", "abrangencia"])["quantidade"].sum()
nao_informado = pd.DataFrame({"total": total, "sem_municipio": sem_municipio})
nao_informado["pct_sem_municipio"] = (nao_informado["sem_municipio"] / nao_informado["total"] * 100).round(2)
nao_informado

## 6. Distribuição por UF (contagem bruta)

Totais de janeiro a outubro por UF para três eventos, filtrando `abrangencia = 'Estadual'`. Para homicídio e feminicídio a contagem é de vítimas; para roubo de veículo, de ocorrências.

**Isto não é um ranking de violência.** SP tem mais casos que RR em parte porque tem muito mais habitantes, e as UFs podem registrar de formas diferentes. As taxas por 100 mil habitantes entram no notebook 02.

In [ ]:
eventos_chave = ["Homicídio doloso", "Feminicídio", "Roubo de veículo"]
estadual = df[(df["abrangencia"] == "Estadual") & (df["evento"].isin(eventos_chave))]
por_uf = estadual.pivot_table(index="uf", columns="evento", values="quantidade", aggfunc="sum").sort_values("Homicídio doloso", ascending=False)
print(por_uf.shape)
print(por_uf.sum())
por_uf

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
por_uf["Homicídio doloso"].plot.bar(ax=ax, color="steelblue")
ax.set_title("Vítimas de homicídio doloso por UF, jan-out 2026 (contagem bruta)")
ax.set_xlabel("")
ax.set_ylabel("vítimas")
plt.tight_layout()
plt.show()

## 7. Distribuição por mês (Brasil)

Série nacional mensal dos mesmos eventos. Os eventos têm escalas diferentes, por isso cada um ganha um gráfico próprio.

Duas ressalvas para a leitura:

- os meses têm de 28 a 31 dias, então parte da variação vem só do calendário;
- os meses mais recentes ainda podem ser revisados pelas UFs.

In [ ]:
por_mes = estadual.pivot_table(index="mes", columns="evento", values="quantidade", aggfunc="sum")
por_mes

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, evento in zip(eixos, eventos_chave):
    por_mes[evento].plot(ax=ax, marker="o", color="steelblue")
    ax.set_title(evento)
    ax.set_xlabel("mês")
    ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## 8. Municípios com o mesmo nome

Mostra quantos nomes de município aparecem em mais de uma UF. Por isso toda análise por município usa o par `(uf, municipio)`, e o LLM vai precisar da mesma regra no prompt.

In [ ]:
pares = df.loc[df["municipio"] != "NÃO INFORMADO", ["uf", "municipio"]].drop_duplicates()
homonimos = pares.groupby("municipio")["uf"].agg(n_ufs="nunique", ufs=lambda s: ", ".join(sorted(s)))
homonimos = homonimos[homonimos["n_ufs"] > 1].sort_values("n_ufs", ascending=False)
print("Nomes em mais de uma UF:", len(homonimos))
print("Municípios envolvidos:", homonimos["n_ufs"].sum())
homonimos.head(15)

In [ ]:
conexao.close()

## 9. Resumo

O que os prints acima mostram:

- **Tamanho**
  - 675.482 linhas, de jan a out de 2026, nas 27 UFs.
  - São 5.570 municípios e mais uma linha "NÃO INFORMADO" por UF, totalizando 5.597 pares.
- **Eventos**
  - Há 31 eventos e 43 combinações de evento e abrangência. Nove eventos aparecem em mais de uma abrangência (Estadual, PF, PRF).
  - Só 12 combinações são abertas por município. As outras 31 existem apenas por UF.
  - Entre as que existem só por UF estão estupro, roubo e furto de veículo, tráfico e apreensões. Esses eventos não podem ser analisados por município.
- **Métrica**
  - 24 combinações contam vítimas, 15 contam um total (ocorrências ou procedimentos) e 4 trazem peso, numa unidade não informada.
  - Somar eventos de métricas diferentes não faz sentido.
- **Sexo da vítima:** onde existe, `feminino + masculino + nao_informado` bate com `quantidade` em todas as 547.835 linhas.
- **Cobertura**
  - MG e RJ não informam apreensão de cocaína e maconha (Estadual).
  - RO não informa os eventos de bombeiros. PA e RO não informam alvarás e vistorias.
  - Os eventos da PRF param em maio ou agosto, e "Mortes no trânsito" da PF só tem 4 meses. É isso que explica as linhas a menos em setembro e outubro.
  - Vários eventos estaduais têm menos UFs nos meses mais recentes.
- **Zeros**
  - Nos eventos municipais, a maioria das linhas é zero: 82% em homicídio doloso e 98% em feminicídio. Taxas de municípios pequenos vão oscilar muito.
- **Sem município:** quase nada fica em "NÃO INFORMADO", exceto mandado de prisão (Estadual), com 1,16% do total.
- **Por UF e por mês (contagem bruta)**
  - Os totais dependem do tamanho da população. Não servem para comparar UFs.
  - A série nacional mensal varia entre os meses, mas este notebook não testa se a variação é sazonal.
- **Homônimos:** 232 nomes de município aparecem em mais de uma UF, o que envolve 505 municípios. Toda análise e todo SQL precisa usar `uf` junto com `municipio`.

Regras que levamos para os próximos notebooks:

- filtrar sempre por `abrangencia`;
- não somar métricas diferentes;
- tratar a ausência de linha como "não informado", não como zero;
- calcular taxas com população (notebook 02).